# 02 · Baseline smoke test

用 `baseline.yml` 派生一个仅 1 epoch 的临时配置，验证完整 `train → test → evaluate`。正式 `baseline.yml` 不会被修改，tracing 保持关闭。此 smoke 仍需要 01 已生成的真实预处理 artifacts。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import yaml

PROJECT_ROOT = Path("/content/orthrus")
DRIVE_ROOT = Path("/content/drive/MyDrive/mstc_pids")
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
DATA_ROOT = DRIVE_ROOT / "data"
DATASET = "THEIA_E3"
SEED = 0
BASE_CONFIG = PROJECT_ROOT / "config/experiments/baseline.yml"
SMOKE_CONFIG = ARTIFACT_ROOT / "environment/smoke_baseline_1epoch.yml"
os.environ["ORTHRUS_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["ORTHRUS_DATA_ROOT"] = str(DATA_ROOT)
assert DATASET in {"THEIA_E3", "THEIA_E5"}
assert BASE_CONFIG.is_file(), BASE_CONFIG

In [ ]:
# 仅写入 artifact metadata 区，不改正式配置。
smoke = yaml.safe_load(BASE_CONFIG.read_text(encoding="utf-8"))
smoke.setdefault("pipeline", {})["run_tracing"] = False
smoke.setdefault("detection", {}).setdefault("gnn_training", {})["num_epochs"] = 1
SMOKE_CONFIG.parent.mkdir(parents=True, exist_ok=True)
SMOKE_CONFIG.write_text(yaml.safe_dump(smoke, sort_keys=False), encoding="utf-8")
print(SMOKE_CONFIG.read_text(encoding="utf-8"))

In [ ]:
command = [
    sys.executable, str(PROJECT_ROOT / "src/experiments/run_experiment.py"),
    "--dataset", DATASET, "--config", str(SMOKE_CONFIG), "--seed", str(SEED),
    "--artifact-root", str(ARTIFACT_ROOT), "--stages", "train,test,evaluate",
]
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

In [ ]:
src_root = str(PROJECT_ROOT / "src")
if src_root not in sys.path:
    sys.path.insert(0, src_root)
from artifact_paths import resolve_run_dir

run_dir = resolve_run_dir(ARTIFACT_ROOT.resolve(), DATASET, "orthrus_baseline", SEED)
required = {
    "environment.json": run_dir / "environment.json",
    "config_resolved.yml": run_dir / "config_resolved.yml",
    "runtime.json": run_dir / "runtime.json",
    "metrics.json": run_dir / "node_scores/metrics.json",
}
checkpoint_files = sorted((run_dir / "checkpoints").rglob("checkpoint.pt"))
for label, path in required.items():
    print(label, path, "OK" if path.is_file() else "MISSING")
missing = [label for label, path in required.items() if not path.is_file()]
if missing or not checkpoint_files:
    raise RuntimeError(f"Smoke artifacts 不完整：missing={missing}, checkpoints={checkpoint_files}")
print("checkpoint:", *checkpoint_files, sep="\n")
print("Baseline smoke 通过：", run_dir)